# Sentiment — Improved Classical ML
Logistic Regression · LinearSVC · TF-IDF with bigrams

In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

## 1. Load Data

In [2]:
train_df = pd.read_csv('../data/clean_train.csv')
val_df   = pd.read_csv('../data/clean_val.csv')

train_df['tweet_content'] = train_df['tweet_content'].astype(str).fillna('')
val_df['tweet_content']   = val_df['tweet_content'].astype(str).fillna('')

train_df = train_df[train_df['tweet_content'].str.strip() != '']
val_df   = val_df[val_df['tweet_content'].str.strip()   != '']

label_names = ['Negative', 'Neutral', 'Positive']
print('Train:', len(train_df), '| Val:', len(val_df))

Train: 59733 | Val: 827


## 2. Improved TF-IDF Vectorizer
Bigrams + more features capture richer context.

In [3]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),          # unigrams + bigrams
    sublinear_tf=True,           # log-scale TF
    min_df=2                     # ignore very rare tokens
)

X_train = vectorizer.fit_transform(train_df['tweet_content'])
X_val   = vectorizer.transform(val_df['tweet_content'])
y_train = train_df['label']
y_val   = val_df['label']

print('Feature matrix shape:', X_train.shape)

Feature matrix shape: (59733, 50000)


## 3. Logistic Regression (tuned)

In [4]:
lr_model = LogisticRegression(
    C=5.0,              # slightly higher regularisation
    max_iter=500,
    solver='saga',      # saga handles large datasets better
    n_jobs=-1
)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_val)
print('=== Logistic Regression ===')
print(f'Val Accuracy: {accuracy_score(y_val, y_pred_lr):.4f}')
print(classification_report(y_val, y_pred_lr, target_names=label_names))

e:\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


=== Logistic Regression ===
Val Accuracy: 0.9794
              precision    recall  f1-score   support

    Negative       0.98      0.98      0.98       265
     Neutral       0.99      0.97      0.98       285
    Positive       0.97      0.98      0.97       277

    accuracy                           0.98       827
   macro avg       0.98      0.98      0.98       827
weighted avg       0.98      0.98      0.98       827



## 4. LinearSVC (often beats LR on text)

In [5]:
svm_model = LinearSVC(
    C=1.0,
    max_iter=2000
)
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_val)
print('=== LinearSVC ===')
print(f'Val Accuracy: {accuracy_score(y_val, y_pred_svm):.4f}')
print(classification_report(y_val, y_pred_svm, target_names=label_names))

=== LinearSVC ===
Val Accuracy: 0.9782
              precision    recall  f1-score   support

    Negative       0.98      0.98      0.98       265
     Neutral       0.98      0.98      0.98       285
    Positive       0.97      0.98      0.97       277

    accuracy                           0.98       827
   macro avg       0.98      0.98      0.98       827
weighted avg       0.98      0.98      0.98       827



## 5. Save Best Models

In [ ]:
joblib.dump(lr_model,   '../models/logistic_regression.pkl')
joblib.dump(svm_model,  '../models/svm_model.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print('Saved.')

Saved.
